In [1]:
import arff
import pandas as pd

# загружаем данныу
with open("c:/Personal/Science/Data/Binary_classification/Bank_Customer_Churn.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
df.head()

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,596,Germany,Male,32,3,96709.07,2,0,0,41788.37,No
1,623,France,Male,43,1,0.00,2,1,1,146379.30,No
2,601,Spain,Female,44,4,0.00,2,1,0,58561.31,No
3,506,Germany,Male,59,8,119152.10,2,1,1,170679.74,No
4,560,Spain,Female,27,7,124995.98,1,1,1,114669.79,No


In [2]:
df.shape

(10000, 11)

In [3]:
df.isnull().sum()
df.isna().sum()


credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [4]:
df.dtypes

credit_score          int64
country              object
gender               object
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card          object
active_member        object
estimated_salary    float64
churn                object
dtype: object

In [5]:
df['credit_card'] = df['credit_card'].astype(int)
df['active_member'] = df['active_member'].astype(int)
df['churn'] = df['churn'].map({'Yes' : 1, 'No' : 0})

In [6]:
df.head()
df.dtypes

credit_score          int64
country              object
gender               object
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card           int32
active_member         int32
estimated_salary    float64
churn                 int64
dtype: object

In [14]:
# максимальное число различных категорий среди всех категориальных столбцов
categorical_cols = ['gender', 'country']
max_categories = df[categorical_cols].nunique().max()
ANOG = df.shape[0]/max_categories
print (ANOG)

3333.3333333333335


In [8]:
y = df["churn"]
X = df.drop(["churn"], axis=1)
X.head()

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,596,Germany,Male,32,3,96709.07,2,0,0,41788.37
1,623,France,Male,43,1,0.00,2,1,1,146379.30
2,601,Spain,Female,44,4,0.00,2,1,0,58561.31
3,506,Germany,Male,59,8,119152.10,2,1,1,170679.74
4,560,Spain,Female,27,7,124995.98,1,1,1,114669.79


In [10]:
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

cat_cols = ['gender', 'country']

encoders = {
    "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
    "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
    "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
    "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
    "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
    "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
    "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols),
    "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough")
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
    "LightGBM": LGBMClassifier(verbose=-1),
    "CatBoost": CatBoostClassifier(verbose=0)
}

results_auc = pd.DataFrame(
    index=models.keys(),
    columns=encoders.keys()
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for enc_name, encoder_factory in encoders.items():
    for model_name, model in models.items():
        
        pipe = Pipeline([
            ("encoder", encoder_factory()),
            ("model", model)
        ])

        scores = cross_validate(pipe,X,y,cv=cv, scoring="roc_auc")
        auc = round(scores['test_score'].mean(), 4)
        results_auc.loc[model_name, enc_name] = auc
        
        print(f"########## {model_name}, {enc_name} ##########")
        print(f"AUC: {auc}")

print(results_auc)

C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documenta

########## LogReg, OneHot ##########
AUC: 0.753
########## CART, OneHot ##########
AUC: 0.6922
########## RF, OneHot ##########
AUC: 0.8532
########## XGBoost, OneHot ##########
AUC: 0.8442
########## LightGBM, OneHot ##########
AUC: 0.8602
########## CatBoost, OneHot ##########
AUC: 0.8666


C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documenta

########## LogReg, Gray ##########
AUC: 0.7584
########## CART, Gray ##########
AUC: 0.6854
########## RF, Gray ##########
AUC: 0.8531
########## XGBoost, Gray ##########
AUC: 0.8442
########## LightGBM, Gray ##########
AUC: 0.8591
########## CatBoost, Gray ##########
AUC: 0.866


C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documenta

########## LogReg, BaseN ##########
AUC: 0.7559
########## CART, BaseN ##########
AUC: 0.6924
########## RF, BaseN ##########
AUC: 0.8546
########## XGBoost, BaseN ##########
AUC: 0.8476
########## LightGBM, BaseN ##########
AUC: 0.8614
########## CatBoost, BaseN ##########
AUC: 0.8668


... (10 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documenta

########## LogReg, Similarity ##########
AUC: 0.7625
########## CART, Similarity ##########
AUC: 0.6915
########## RF, Similarity ##########
AUC: 0.8531
########## XGBoost, Similarity ##########
AUC: 0.8445


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


########## LightGBM, Similarity ##########
AUC: 0.8602
########## CatBoost, Similarity ##########
AUC: 0.867
          OneHot    Gray   BaseN Hashing Helmert     Sum BackwardDifference  \
LogReg     0.753  0.7584  0.7559  0.7554  0.7478  0.7489             0.7559   
CART      0.6922  0.6854  0.6924  0.6922  0.6918   0.694             0.6912   
RF        0.8532  0.8531  0.8546  0.8489  0.8539  0.8533             0.8552   
XGBoost   0.8442  0.8442  0.8476  0.8445  0.8442  0.8442             0.8439   
LightGBM  0.8602  0.8591  0.8614  0.8602  0.8599  0.8602             0.8607   
CatBoost  0.8666   0.866  0.8668  0.8674  0.8666  0.8676             0.8671   

         MinHash Similarity  
LogReg     0.645     0.7625  
CART      0.6908     0.6915  
RF        0.8464     0.8531  
XGBoost   0.8445     0.8445  
LightGBM  0.8604     0.8602  
CatBoost   0.867      0.867  


In [12]:
results_auc.to_csv("results_auc.csv", index=True)